In [1]:
import os
import re
import pandas as pd
import json

In [2]:
folder = 'data'

In [3]:
def read_ann_files(directory):
    ann_files = [f for f in os.listdir(directory) if f.endswith('.ann')]
    data = {}
    for file in ann_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            data[file] = f.read()
    return data

def extract_text_and_offsets(details):
    parts = details.split('\t')
    entity_info = parts[0].split(' ')
    text_content = parts[-1]
    offsets = ' '.join([part for part in entity_info[1:] if part.isdigit()])
    return text_content, offsets

def parse_ann_file(content):
    lines = content.strip().split('\n')
    df = pd.DataFrame([line.split('\t', 1) for line in lines], columns=['ID', 'Details'])
    df['Text'], df['Offsets'] = zip(*df['Details'].apply(extract_text_and_offsets))
    t_e_df = df[df['ID'].str.startswith(('T', 'E'))]
    text_dict = t_e_df.set_index('ID')['Text'].to_dict()
    offset_dict = t_e_df.set_index('ID')['Offsets'].to_dict()
    rel_df = df[df['ID'].str.startswith('R')]
    
    # Für die normalen Fälle 
    rel_pattern = re.compile(r'^(R\d+)\t(And|Or) Arg1:(E\d+|T\d+) Arg2:(E\d+|T\d+)$')
    # Für alle Fälle mit AND,OR die eigentlich keinen Sinn machen
    ##rel_pattern = re.compile(r'^R\d+\t.*?(And|Or).*$')
    
    relationships = []
    for index, row in rel_df.iterrows():
        line = f"{row['ID']}\t{row['Details']}"
        rel_match = rel_pattern.match(line)
        if rel_match:
            rel_id, rel_type, arg1, arg2 = rel_match.groups()
            relationships.append((rel_type, arg1, arg2))
    neg_df = df[df['Details'].str.contains('Negation')]
    negations = neg_df[['ID', 'Text', 'Offsets']].to_dict(orient='records')
    return text_dict, offset_dict, relationships, negations

def get_full_text(text_dict, offset_dict, entity_id):
    if entity_id in text_dict:
        text_content = text_dict[entity_id]
        offsets = offset_dict[entity_id]
        if entity_id.startswith('E'):
            sub_entity_id = re.search(r'\b(T\d+)\b', text_content)
            if sub_entity_id:
                sub_text, sub_offsets = get_full_text(text_dict, offset_dict, sub_entity_id.group(1))
                return sub_text, sub_offsets
        return text_content, offsets
    return entity_id, ""

def create_rel_texts(text_dict, offset_dict, relationships, negations):
    rel_texts = []
    for rel_type, arg1, arg2 in relationships:
        arg1_text, arg1_offset = get_full_text(text_dict, offset_dict, arg1)
        arg2_text, arg2_offset = get_full_text(text_dict, offset_dict, arg2)
        rel_texts.append({
            "type": rel_type,
            "arg1_text": arg1_text,
            "arg1_offset": arg1_offset,
            "arg2_text": arg2_text,
            "arg2_offset": arg2_offset
        })
    for neg in negations:
        if 'Negation' in neg['Text']:
            continue
        rel_texts.append({
            "type": "Negation",
            "arg1_text": neg['Text'],
            "arg1_offset": neg['Offsets'],
            "arg2_text": "",
            "arg2_offset": ""
        })
    return rel_texts

def save_intermediate_results(rel_texts, file_prefix, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    file_name = f"{file_prefix}.json"
    file_path = os.path.join(output_directory, file_name)
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(rel_texts, f, indent=4)


def read_txt_files(directory):
    txt_files = [f for f in os.listdir(directory) if f.endswith('.txt')]
    data = {}
    for file in txt_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            data[file] = f.read()
    return data

def insert_relations_to_text(original_text, relations):
    new_text = original_text
    relation_insertions = []

    for rel in relations:
        if rel['type'].lower() == 'negation' and rel['arg1_offset']:
            start_offset = int(rel['arg1_offset'].split(' ')[0])
            relation_str = f"[NOT] {rel['arg1_text']}"
            relation_insertions.append((start_offset, relation_str, len(rel['arg1_text'])))
        else:
            arg1_end_offset = int(rel['arg1_offset'].split(' ')[-1])
            relation_str = f" [{rel['type'].upper()}] "
            relation_insertions.append((arg1_end_offset + 1, relation_str, 0))

    # Sort offsets and perform insertions
    for insert_pos, relation_str, original_len in sorted(relation_insertions, reverse=True):
        new_text = new_text[:insert_pos] + relation_str + new_text[insert_pos + original_len:]

    return new_text

def write_new_files(new_directory, data, all_rel_texts):
    if not os.path.exists(new_directory):
        os.makedirs(new_directory)
    for file, content in data.items():
        ann_file = file.replace('.txt', '.ann')
        if ann_file in all_rel_texts:
            rel_texts = all_rel_texts[ann_file]
            new_text = insert_relations_to_text(content, rel_texts)
            if new_text is not None:
                file_path = os.path.join(new_directory, file)
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(new_text)

In [ ]:
directory = folder
output_directory = f'{folder}_entitys'
data = read_ann_files(directory)

all_rel_texts = {}
for file, content in data.items():
    text_dict, offset_dict, relationships, negations = parse_ann_file(content)
    rel_texts = create_rel_texts(text_dict, offset_dict, relationships, negations)
    file_prefix = os.path.splitext(file)[0]
    save_intermediate_results(rel_texts, file_prefix, output_directory)
    all_rel_texts[file] = rel_texts

In [ ]:
txt_directory = folder
new_directory = f'{folder}_parsed_1'
txt_data = read_txt_files(txt_directory)
write_new_files(new_directory, txt_data, all_rel_texts)


In [4]:
# wrong order
# NCT03861689 
